It follows the same steps as previous one, only the used dataset is different.

In [ ]:
%cd "/content/drive/MyDrive/PhD_Documents/IITG-Classes/NLP-with-LLM/MultiClinSum"

/content/drive/MyDrive/PhD_Documents/IITG-Classes/NLP-with-LLM/MultiClinSum


In [ ]:
%ls

1-Extractive-Summarizer-on-592-gs-train-en.ipynb
2-Extractive-Sumamrizer-on-test-en
multiclinsum_gs_train_en/
multiclinsum_large-scale_train_en/
multiclinsum_test_en/
sentence_segmented_gs_train_en.csv


In [ ]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "multiclinsum_test_en/"
)

FULLTEXT_DIR = BASE_DIR / "fulltext"
SUMMARY_DIR = BASE_DIR / "summaries"

# Load full-text reports
reports = {
    path.stem: path.read_text(
        encoding="utf-8",
        errors="replace"
    ).strip()
    for path in FULLTEXT_DIR.glob("*.txt")
}

# Load summaries, removing the "_sum" suffix
summaries = {
    path.stem.removesuffix("_sum"): path.read_text(
        encoding="utf-8",
        errors="replace"
    ).strip()
    for path in SUMMARY_DIR.glob("*.txt")
}

# Match reports and summaries by their base filename
common_ids = sorted(
    set(reports.keys()) & set(summaries.keys())
)

df = pd.DataFrame([
    {
        "filename": file_id + ".txt",
        "report": reports[file_id],
        "summary": summaries[file_id]
    }
    for file_id in common_ids
])

print("Full-text files:", len(reports))
print("Summary files:", len(summaries))
print("Matched pairs:", len(df))

display(df.head(3))


Full-text files: 3396
Summary files: 3396
Matched pairs: 3396


,filename,report,summary
0,multiclinsum_test_1000_en.txt,"On July 10, 2022, a 40-year-old woman was admi...",We report the case of a 40-year-old woman with...
1,multiclinsum_test_1001_en.txt,A 38-year-old woman developed numbness in the ...,We found a 38-year-old female patient with KFS...
2,multiclinsum_test_1002_en.txt,"In July 2006, a 41-year-old female presented w...",We present a case of a 41-year-old female that...


In [ ]:
# Check for unmatched files


unmatched_reports = set(reports) - set(summaries)
unmatched_summaries = set(summaries) - set(reports)

print("Unmatched reports:", len(unmatched_reports))
print("Unmatched summaries:", len(unmatched_summaries))

print("Empty reports:", (df["report"].str.len() == 0).sum())
print("Empty summaries:", (df["summary"].str.len() == 0).sum())

assert len(df) > 0, "No matching report-summary pairs found!"


Unmatched reports: 0
Unmatched summaries: 0
Empty reports: 0
Empty summaries: 0


In [ ]:

# # Inspect the exact unmatched files

# print("=== UNMATCHED FULL-TEXT REPORTS ===")
# for file_id in sorted(unmatched_reports):
#     print(repr(file_id))
#     print("File:", FULLTEXT_DIR / f"{file_id}.txt")

# print("\n=== UNMATCHED SUMMARIES ===")
# for file_id in sorted(unmatched_summaries):
#     print(repr(file_id))
#     print("File:", SUMMARY_DIR / f"{file_id}_sum.txt")




In [ ]:
# duplicate_path = FULLTEXT_DIR / "multiclinsum_test_1351_en (1).txt"

# if duplicate_path.exists():
#     duplicate_path.unlink()
#     print("Duplicate file deleted:", duplicate_path)
# else:
#     print("File not found:", duplicate_path)

Duplicate file deleted: multiclinsum_test_en/fulltext/multiclinsum_test_1351_en (1).txt


# Pre-processing

In [ ]:

import re
import pandas as pd

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Convert all characters to lowercase
    text = text.lower()

    # 2. Replace decimal points with a temporary placeholder
    #    Example: 5.3 -> 5DOT3
    text = re.sub(
        r'(?<!\w)(\d+)\.(\d+)(?!\w)',
        r'\1DOT\2',
        text
    )

    # 3. Standardize phrases
    text = text.replace("and/or", "and or")

    # 4. Separate hyphenated number-word patterns
    # Examples: 52-year-old -> 52 year old
    #           10-day history -> 10 day history
    #           3-month follow-up -> 3 month follow up
    text = re.sub(
        r'(?<=\d)-(?=[a-z])|(?<=[a-z])-(?=[a-z])',
        ' ',
        text
    )

    # 5. Remove unnecessary spaces and normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# save a new dataframe
cleaned_df = df.copy(deep=True)

# Apply preprocessing
cleaned_df["cleaned_report"] = df["report"].apply(preprocess_text)
cleaned_df["cleaned_summary"] = df["summary"].apply(preprocess_text)

print("Preprocessing completed!")


Preprocessing completed!


In [ ]:
df.head(2)

,filename,report,summary
0,multiclinsum_test_1000_en.txt,"On July 10, 2022, a 40-year-old woman was admi...",We report the case of a 40-year-old woman with...
1,multiclinsum_test_1001_en.txt,A 38-year-old woman developed numbness in the ...,We found a 38-year-old female patient with KFS...


In [ ]:
cleaned_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary
0,multiclinsum_test_1000_en.txt,"On July 10, 2022, a 40-year-old woman was admi...",We report the case of a 40-year-old woman with...,"on july 10, 2022, a 40 year old woman was admi...",we report the case of a 40 year old woman with...
1,multiclinsum_test_1001_en.txt,A 38-year-old woman developed numbness in the ...,We found a 38-year-old female patient with KFS...,a 38 year old woman developed numbness in the ...,we found a 38 year old female patient with kfs...


# Sentence Segmentation

In [ ]:

!pip -q install nltk

import nltk
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# # Testing on one sample

# # Select one example report
# processed_text = cleaned_df["cleaned_report"][0]

# sentences = sent_tokenize(processed_text, language="english")

# print("Number of sentences:", len(sentences))

# for i, sentence in enumerate(sentences, start=1):
#     print(f"{i}. {sentence}")


Number of sentences: 39
1. on july 10, 2022, a 40 year old woman was admitted to the first affiliated hospital of xi’an jiaotong university due to intermittent fever for 2 months.
2. two months prior, her temperature fluctuated between 37DOT4 and 38DOT4°c in the afternoons, and chills and a dry cough accompanied this symptom.
3. she had previously undergone examination and symptomatic treatment at a regional hospital.
4. however, the treatment was ineffective.
5. furthermore, arthralgia and fatigue gradually appeared, with no chest pain and tightness, or rash.
6. a month and a half prior, she was diagnosed with depression and treated with escitalopram oxalate, and magnesium valproate sustained release tablets at a hospital.
7. the symptoms slightly improved after treatment.
8. half a month prior, a b ultrasonic examination showed lymphadenopathy throughout her body.
9. therefore, lymphoma was suspected but she did not get any treatment.
10. one day prior to her arrival at our hospital,

In [ ]:

sent_seg_df = cleaned_df.copy(deep=True)

# Apply sentence segmentation to all reports and sumamries
sent_seg_df["r_sentences"] = cleaned_df["cleaned_report"].apply(
    lambda text: sent_tokenize(text, language="english")
)

sent_seg_df["su_sentences"] = cleaned_df["cleaned_summary"].apply(
    lambda text: sent_tokenize(text, language="english")
)

sent_seg_df["r_sent_count"] = sent_seg_df["r_sentences"].apply(len)
sent_seg_df["su_sent_count"] = sent_seg_df["su_sentences"].apply(len)

print("Average sentences per report:", sent_seg_df["r_sent_count"].mean())
print("Maximum sentences:", sent_seg_df["r_sent_count"].max())
print("Minimum sentences:", sent_seg_df["r_sent_count"].min())

print("Average sentences per summary:", sent_seg_df["su_sent_count"].mean())
print("Maximum sentences:", sent_seg_df["su_sent_count"].max())
print("Minimum sentences:", sent_seg_df["su_sent_count"].min())


Average sentences per report: 27.522673733804474
Maximum sentences: 259
Minimum sentences: 3
Average sentences per summary: 5.358951707891637
Maximum sentences: 22
Minimum sentences: 1


In [ ]:
sent_seg_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count
0,multiclinsum_test_1000_en.txt,"On July 10, 2022, a 40-year-old woman was admi...",We report the case of a 40-year-old woman with...,"on july 10, 2022, a 40 year old woman was admi...",we report the case of a 40 year old woman with...,"[on july 10, 2022, a 40 year old woman was adm...",[we report the case of a 40 year old woman wit...,39,5
1,multiclinsum_test_1001_en.txt,A 38-year-old woman developed numbness in the ...,We found a 38-year-old female patient with KFS...,a 38 year old woman developed numbness in the ...,we found a 38 year old female patient with kfs...,[a 38 year old woman developed numbness in the...,[we found a 38 year old female patient with kf...,33,8


In [ ]:
sent_seg_df["su_sentences"][0]

['we report the case of a 40 year old woman with intermittent fever for 2 months and gradually accompanied by chills, dry cough, arthralgia, and fatigue.',
 'the patient was also diagnosed with depression after fever.',
 'she received symptomatic treatment at a regional hospital; however, there was no significant symptom relief.',
 'she suddenly developed hemoptysis 1 day prior to arrival at our hospital, where we discovered that her liver, spleen, neck, and axillary lymph nodes were enlarged, and there were multiple nodules in both lungs.',
 'the patient was eventually diagnosed with brucellosis after the serum agglutination test and received antibiotic therapy, which provided symptom relief.']

In [ ]:
# Saving the latest df: sent_seg_df

from pathlib import Path

# Create an output folder in Google Drive
OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the dataframe
csv_path = OUTPUT_DIR / "sentence_segmented_test_en.csv"

sent_seg_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8"
)

print("Saved successfully!")
print("File path:", csv_path)
print("Number of reports:", len(sent_seg_df))

Saved successfully!
File path: /content/drive/MyDrive/PhD_Documents/IITG-Classes/NLP-with-LLM/MultiClinSum/sentence_segmented_test_en.csv
Number of reports: 3396


In [ ]:
import pandas as pd
sent_seg_df = pd.read_csv("/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/sentence_segmented_test_en.csv")


In [ ]:
sent_seg_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count
0,multiclinsum_test_1000_en.txt,"On July 10, 2022, a 40-year-old woman was admi...",We report the case of a 40-year-old woman with...,"on july 10, 2022, a 40 year old woman was admi...",we report the case of a 40 year old woman with...,"['on july 10, 2022, a 40 year old woman was ad...",['we report the case of a 40 year old woman wi...,39,5
1,multiclinsum_test_1001_en.txt,A 38-year-old woman developed numbness in the ...,We found a 38-year-old female patient with KFS...,a 38 year old woman developed numbness in the ...,we found a 38 year old female patient with kfs...,['a 38 year old woman developed numbness in th...,['we found a 38 year old female patient with k...,33,8


In [ ]:
# Load pretrained BERT

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bert-base-multilingual-cased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Device:", device)
print("BERT model loaded:", MODEL_NAME)


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda
BERT model loaded: bert-base-multilingual-cased


# BERT Sentence Embeddings Generation

In [ ]:

def encode_sentences(sentences, batch_size=16, max_length=256):
    """
    Convert sentences into BERT embeddings.

    Returns:
        NumPy array of shape (number_of_sentences, 768)
    """
    if not sentences:
        return np.empty((0, model.config.hidden_size), dtype=np.float32)

    all_embeddings = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():
            outputs = model(**inputs)
            token_embeddings = outputs.last_hidden_state

            # Expand attention mask to match embedding dimensions
            mask = inputs["attention_mask"].unsqueeze(-1)
            mask = mask.to(token_embeddings.dtype)

            # Mean pooling, excluding padding tokens
            summed = (token_embeddings * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1)

            sentence_embeddings = summed / counts

        all_embeddings.append(
            sentence_embeddings.cpu().numpy()
        )

    return np.concatenate(all_embeddings, axis=0)


In [ ]:
import ast

sent_seg_df["r_sentences"] = sent_seg_df["r_sentences"].apply(ast.literal_eval)
sent_seg_df["su_sentences"] = sent_seg_df["su_sentences"].apply(ast.literal_eval)


In [ ]:
print(type(sent_seg_df["r_sentences"][0]))
print(len(sent_seg_df["r_sentences"][0]))
print(sent_seg_df.loc[0, "r_sentences"][0])

<class 'list'>
39
on july 10, 2022, a 40 year old woman was admitted to the first affiliated hospital of xi’an jiaotong university due to intermittent fever for 2 months.


In [ ]:
# Test on one report first

sample_sentences = sent_seg_df["r_sentences"][0]

sample_embeddings = encode_sentences(sample_sentences)

print("Number of sentences:", len(sample_sentences))
print("Embedding shape:", sample_embeddings.shape)
print("Embedding dtype:", sample_embeddings.dtype)
print("First sentence:", sample_sentences[0])
print("First embedding (first 10 values):", sample_embeddings[0][:10])


Number of sentences: 39
Embedding shape: (39, 768)
Embedding dtype: float32
First sentence: on july 10, 2022, a 40 year old woman was admitted to the first affiliated hospital of xi’an jiaotong university due to intermittent fever for 2 months.
First embedding (first 10 values): [ 0.09350358  0.24973705 -0.01328884 -0.33911753  0.2386631   0.05912483
 -0.27514663  0.28585598 -0.28431052  0.0171039 ]


In [ ]:
# Generate embeddings for all the reports

# separate dataframe
bert_df = sent_seg_df.copy(deep=True)

all_report_embeddings = []

for i, sentences in enumerate(bert_df["r_sentences"]):
    embeddings = encode_sentences(
        sentences,
        batch_size=16,
        max_length=256
    )

    all_report_embeddings.append(embeddings)

    if (i + 1) % 100 == 0 or i == len(bert_df) - 1:
        print(f"Processed {i + 1}/{len(bert_df)} reports")

bert_df["sentence_embeddings"] = all_report_embeddings

print("\nEmbedding generation completed!")
print("Total reports:", len(bert_df))
print("First report embedding shape:", bert_df["sentence_embeddings"][0].shape)


Processed 100/3396 reports
Processed 200/3396 reports
Processed 300/3396 reports
Processed 400/3396 reports
Processed 500/3396 reports
Processed 600/3396 reports
Processed 700/3396 reports
Processed 800/3396 reports
Processed 900/3396 reports
Processed 1000/3396 reports
Processed 1100/3396 reports
Processed 1200/3396 reports
Processed 1300/3396 reports
Processed 1400/3396 reports
Processed 1500/3396 reports
Processed 1600/3396 reports
Processed 1700/3396 reports
Processed 1800/3396 reports
Processed 1900/3396 reports
Processed 2000/3396 reports
Processed 2100/3396 reports
Processed 2200/3396 reports
Processed 2300/3396 reports
Processed 2400/3396 reports
Processed 2500/3396 reports
Processed 2600/3396 reports
Processed 2700/3396 reports
Processed 2800/3396 reports
Processed 2900/3396 reports
Processed 3000/3396 reports
Processed 3100/3396 reports
Processed 3200/3396 reports
Processed 3300/3396 reports
Processed 3396/3396 reports

Embedding generation completed!
Total reports: 3396
Firs

In [ ]:
bert_df["sentence_embeddings"][0]

array([[ 0.09350358,  0.24973705, -0.01328884, ...,  0.6372403 ,
         0.08370141, -0.12402735],
       [-0.2874483 , -0.36514854,  0.64538634, ...,  0.08657411,
         0.44785407, -0.31574374],
       [-0.01469565,  0.06046185,  0.09783208, ...,  0.54008657,
        -0.04606109,  0.08165258],
       ...,
       [-0.4239066 , -0.7751326 , -0.21418346, ...,  0.7487283 ,
        -0.25342435,  0.52831423],
       [-0.44598383, -0.28700432,  0.5936671 , ...,  0.24342099,
         0.13824418, -0.17076704],
       [-0.17011964, -0.40078178,  0.27504063, ...,  0.61196995,
         0.696006  , -0.18939336]], dtype=float32)

# K-means Clustering

In [ ]:

from sklearn.cluster import KMeans
import numpy as np

def cluster_and_select_sentences(sentences, embeddings, n_clusters=10):
    """
    Select representative sentences using K-means centroids.

    Returns selected sentences in original document order.
    """
    n_sentences = len(sentences)

    if n_sentences == 0:
        return []

    # Handle reports containing fewer than 10 sentences
    k = min(n_clusters, n_sentences)

    # If K equals the number of sentences, retain all sentences
    if k == n_sentences:
        return list(sentences)

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(embeddings)
    centroids = kmeans.cluster_centers_

    selected_indices = []

    for cluster_id in range(k):
        # Find sentence indices belonging to this cluster
        cluster_indices = np.where(labels == cluster_id)[0]

        if len(cluster_indices) == 0:
            continue

        # Select the sentence nearest to the cluster centroid
        cluster_embeddings = embeddings[cluster_indices]
        distances = np.linalg.norm(
            cluster_embeddings - centroids[cluster_id],
            axis=1
        )

        closest_index = cluster_indices[np.argmin(distances)]
        selected_indices.append(closest_index)

    # Restore original document order
    selected_indices = sorted(set(selected_indices))

    return [sentences[i] for i in selected_indices]


In [ ]:
bert_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count,sentence_embeddings
0,multiclinsum_test_1000_en.txt,"On July 10, 2022, a 40-year-old woman was admi...",We report the case of a 40-year-old woman with...,"on july 10, 2022, a 40 year old woman was admi...",we report the case of a 40 year old woman with...,"[on july 10, 2022, a 40 year old woman was adm...",[we report the case of a 40 year old woman wit...,39,5,"[[0.09350358, 0.24973705, -0.013288841, -0.339..."
1,multiclinsum_test_1001_en.txt,A 38-year-old woman developed numbness in the ...,We found a 38-year-old female patient with KFS...,a 38 year old woman developed numbness in the ...,we found a 38 year old female patient with kfs...,[a 38 year old woman developed numbness in the...,[we found a 38 year old female patient with kf...,33,8,"[[-0.62967557, -0.61783344, 0.22319503, 0.0828..."


In [ ]:

# Test on the first report
sample_sentences = bert_df["r_sentences"][0]
sample_embeddings = bert_df["sentence_embeddings"][0]

sample_selected = cluster_and_select_sentences(
    sample_sentences,
    sample_embeddings,
    n_clusters=10
)

print("Original sentence count:", len(sample_sentences))
print("Selected sentence count:", len(sample_selected))

print("\nExtractive summary:")
print(sample_selected)

Original sentence count: 39
Selected sentence count: 10

Extractive summary:
['on july 10, 2022, a 40 year old woman was admitted to the first affiliated hospital of xi’an jiaotong university due to intermittent fever for 2 months.', 'however, the treatment was ineffective.', 'a month and a half prior, she was diagnosed with depression and treated with escitalopram oxalate, and magnesium valproate sustained release tablets at a hospital.', 'in addition, she had lost 5 kg over the past 3 months.', 'chest computed tomography (ct) revealed multiple pulmonary nodules in both lungs, with a small amount of pleural effusion and enlarged lymph nodes in the bilateral axilla.', 'furthermore, the blood culture for brucella melitensis was positive on the tenth day after admission.', 'characteristic rod shaped gram negative bacteria could be observed under a microscope.', 'subsequently, the patient was definitively diagnosed with brucellosis.', 'following the brucellosis diagnosis, she received ant

In [ ]:
# Generate summaries for all 592 reports

# new dataframe
cluster_df = bert_df.copy(deep=True)

cluster_df["selected_sentences"] = [
    cluster_and_select_sentences(
        sentences,
        embeddings,
        n_clusters=10
    )
    for sentences, embeddings in zip(
        cluster_df["r_sentences"],
        cluster_df["sentence_embeddings"]
    )
]

cluster_df["generated_summary"] = (
    cluster_df["selected_sentences"].apply(" ".join)
)

print("Summaries generated:", len(cluster_df))
print("\nFirst generated summary:\n")
print(cluster_df["generated_summary"][0])


Summaries generated: 3396

First generated summary:

on july 10, 2022, a 40 year old woman was admitted to the first affiliated hospital of xi’an jiaotong university due to intermittent fever for 2 months. however, the treatment was ineffective. a month and a half prior, she was diagnosed with depression and treated with escitalopram oxalate, and magnesium valproate sustained release tablets at a hospital. in addition, she had lost 5 kg over the past 3 months. chest computed tomography (ct) revealed multiple pulmonary nodules in both lungs, with a small amount of pleural effusion and enlarged lymph nodes in the bilateral axilla. furthermore, the blood culture for brucella melitensis was positive on the tenth day after admission. characteristic rod shaped gram negative bacteria could be observed under a microscope. subsequently, the patient was definitively diagnosed with brucellosis. following the brucellosis diagnosis, she received antibiotic therapy with rifampicin (600 mg/dose, once

In [ ]:
cluster_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count,sentence_embeddings,selected_sentences,generated_summary
0,multiclinsum_test_1000_en.txt,"On July 10, 2022, a 40-year-old woman was admi...",We report the case of a 40-year-old woman with...,"on july 10, 2022, a 40 year old woman was admi...",we report the case of a 40 year old woman with...,"[on july 10, 2022, a 40 year old woman was adm...",[we report the case of a 40 year old woman wit...,39,5,"[[0.09350358, 0.24973705, -0.013288841, -0.339...","[on july 10, 2022, a 40 year old woman was adm...","on july 10, 2022, a 40 year old woman was admi..."
1,multiclinsum_test_1001_en.txt,A 38-year-old woman developed numbness in the ...,We found a 38-year-old female patient with KFS...,a 38 year old woman developed numbness in the ...,we found a 38 year old female patient with kfs...,[a 38 year old woman developed numbness in the...,[we found a 38 year old female patient with kf...,33,8,"[[-0.62967557, -0.61783344, 0.22319503, 0.0828...","[half an hour later, she felt numb and weak in...","half an hour later, she felt numb and weak in ..."


In [ ]:
type(cluster_df["cleaned_summary"][0])

str

In [ ]:
type(cluster_df["generated_summary"][0])

str

In [ ]:
type(cluster_df["su_sentences"][0])

list

In [ ]:
print("Generated summary:")
print(cluster_df["generated_summary"][0])

print("\nHuman-written reference summary:")
print(cluster_df["cleaned_summary"][0])

Generated summary:
on july 10, 2022, a 40 year old woman was admitted to the first affiliated hospital of xi’an jiaotong university due to intermittent fever for 2 months. however, the treatment was ineffective. a month and a half prior, she was diagnosed with depression and treated with escitalopram oxalate, and magnesium valproate sustained release tablets at a hospital. in addition, she had lost 5 kg over the past 3 months. chest computed tomography (ct) revealed multiple pulmonary nodules in both lungs, with a small amount of pleural effusion and enlarged lymph nodes in the bilateral axilla. furthermore, the blood culture for brucella melitensis was positive on the tenth day after admission. characteristic rod shaped gram negative bacteria could be observed under a microscope. subsequently, the patient was definitively diagnosed with brucellosis. following the brucellosis diagnosis, she received antibiotic therapy with rifampicin (600 mg/dose, once a day) and doxycycline (100 mg/do

In [ ]:
# Install evaluation libraries

!pip -q install rouge-score bert-score evaluate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer

# Use the existing test predictions.
predictions = cluster_df["generated_summary"].fillna("").astype(str).tolist()

# IMPORTANT:
# Use the reference summaries in the SAME text representation
# as the summaries used for model training/evaluation.
references = cluster_df["cleaned_summary"].fillna("").astype(str).tolist()

assert len(predictions) == len(references)
assert len(predictions) > 0

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

all_scores = []

for prediction, reference in zip(predictions, references):
    # rouge_score expects reference first, prediction second.
    scores = scorer.score(reference, prediction)

    row = {}

    for metric in ["rouge1", "rouge2", "rougeL"]:
        row[f"{metric}_precision"] = scores[metric].precision
        row[f"{metric}_recall"] = scores[metric].recall
        row[f"{metric}_f1"] = scores[metric].fmeasure

    all_scores.append(row)

rouge_per_example = pd.DataFrame(all_scores)

# Mean of the individual example scores, on a 0–100 scale.
rouge_summary = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
    ],
    "Precision": [
        rouge_per_example["rouge1_precision"].mean() * 100,
        rouge_per_example["rouge2_precision"].mean() * 100,
        rouge_per_example["rougeL_precision"].mean() * 100
    ],
    "Recall": [
        rouge_per_example["rouge1_recall"].mean() * 100,
        rouge_per_example["rouge2_recall"].mean() * 100,
        rouge_per_example["rougeL_recall"].mean() * 100
    ],
    "F1": [
        rouge_per_example["rouge1_f1"].mean() * 100,
        rouge_per_example["rouge2_f1"].mean() * 100,
        rouge_per_example["rougeL_f1"].mean() * 100
    ]
})

print("ROUGE RESULTS (mean per-example scores, %)")
display(rouge_summary.round(4))

ROUGE RESULTS (mean per-example scores, %)


,Metric,Precision,Recall,F1
0,ROUGE-1,28.4404,53.6186,35.3496
1,ROUGE-2,11.8339,21.4321,14.5035
2,ROUGE-L,17.9631,34.1324,22.3614


In [ ]:
import evaluate

bertscore = evaluate.load("bertscore")

bert_results = bertscore.compute(
    predictions=predictions,
    references=references,
    model_type="roberta-large",
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=4,
    rescale_with_baseline=False
)

bert_summary = pd.DataFrame({
    "Metric": ["BERTScore"],
    "Precision": [np.mean(bert_results["precision"]) * 100],
    "Recall": [np.mean(bert_results["recall"]) * 100],
    "F1": [np.mean(bert_results["f1"]) * 100]
})

print("BERTSCORE RESULTS (mean per-example scores, %)")
display(bert_summary.round(4))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTSCORE RESULTS (mean per-example scores, %)


,Metric,Precision,Recall,F1
0,BERTScore,82.8203,86.8161,84.7515
